# 06 — Full Pipeline & Publication Figures
## Cross-Subject Validation and the Three Figures

---

You have now built every component. This notebook assembles them into:

1. **A cross-subject pipeline** — run Modules 1-5 on all 4 subjects and aggregate results
2. **Three publication-quality figures** — the scientific result
3. **Statistical testing** — making the result credible
4. **The electrode capacity experiment** — connecting to information theory

---

## The Three Figures

### Figure 1 — Latent Trajectory Geometry
**What it shows:** Sequential VAE latent trajectories in 3D PCA space, for representative subject and condition.  
**Scientific claim:** Population activity traces distinct, structured paths in latent space for different WM loads. The geometry is richer at higher load.  
**Key design decision:** Show both mean trajectory (thick line) and individual trials (thin lines). Plot all 3 conditions side by side. Mark stimulus onset with a marker.

### Figure 2 — Geometric Biomarkers Over Time
**What it shows:** Two-panel time course:
- Panel A: θ_min(t) for 0-back vs 2-back (load-dependent separation)
- Panel B: Q(t) for target vs non-target in 2-back (detection-related instability)
  
**Scientific claim:** Both metrics capture distinct aspects of WM dynamics. Subspace collapse tracks representational interference; tangling tracks dynamical instability.  
**Key design:** Error bands = SEM across subjects (n=4). Mark the stimulus onset. Shade the maintenance window.

### Figure 3 — LQR Rescue
**What it shows:** State-space panel + energy-accuracy curve, for representative high-tangling trial.  
**Scientific claim:** A minimum-energy control signal exists that would have stabilized the failing trajectory. The energy requirement scales with geometric collapse severity.  
**Key design:** Show uncontrolled (failing) and controlled trajectories. Show the Pareto curve (energy vs. state error) for multiple R_c values.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import sys
from pathlib import Path

sys.path.insert(0, str(Path('../../scripts').resolve()))
from ieeg_utils import (
    SUBJECTS, load_subject, preprocess, high_gamma_power, epoch_data,
    baseline_normalize, reject_bad_channels, pca_project,
    principal_angles, trajectory_tangling
)

plt.rcParams.update({
    'font.size': 12,
    'axes.spines.right': False,
    'axes.spines.top': False,
    'figure.dpi': 150
})

In [ ]:
# ─── Cross-subject pipeline ────────────────────────────────────────────────────
# Run full preprocessing + VAE + geometry for each subject
# Store results in a dict indexed by subject

# Template structure — fill in as you complete Modules 1-5:
#
# results = {}  # subj -> dict of results
#
# for subj in SUBJECTS:
#     print(f'Processing {subj}...')
#
#     # Module 1: preprocess
#     d        = load_subject(subj)
#     good_ch  = reject_bad_channels(d['data'])
#     data_c   = preprocess(d['data'][:, good_ch])
#     hgp      = high_gamma_power(data_c)
#     ep_dict  = epoch_data(hgp, d['stim'], d['task'], d['target'])
#     epochs   = baseline_normalize(ep_dict['epochs'], ep_dict['times'])
#
#     # Module 2: VAE latents (requires trained model — save/load from Module 2)
#     # mu_np = load or compute latent trajectories
#
#     # Module 3: geometry
#     # t_centers, angles = time_resolved_angles(...)
#     # Q = trial_tangling(...)
#
#     # Module 4: DMD eigenspectrum
#
#     # Module 5: LQR rescue
#
#     results[subj] = {'angles': ..., 'tangling': ..., 'eigs': ..., 'rescue': ...}

print('Cross-subject pipeline template ready.')
print('Run this after completing Modules 1-5 for subject al.')

In [ ]:
# ─── Figure 1: Latent Trajectory Geometry ────────────────────────────────────
#
# Publication quality: 7 inches wide (Nature column = 3.5", 2-col = 7")
# Colors: muted, colorblind-friendly
#
# Template:
#
# PALETTE = {'zero_back': '#4878CF', 'one_back': '#6ACC65', 'two_back': '#D65F5F'}
#
# fig = plt.figure(figsize=(7, 3))
# for col, (cond, name) in enumerate([(0,'zero_back'), (1,'one_back'), (2,'two_back')]):
#     ax = fig.add_subplot(1, 3, col+1, projection='3d')
#     mask = task_id == cond
#
#     # Individual trials (thin, transparent)
#     for ti in np.where(mask)[0]:
#         ax.plot(*scores_3d[ti].T, color=PALETTE[name], lw=0.3, alpha=0.2)
#
#     # Mean trajectory (thick)
#     mean_traj = scores_3d[mask].mean(axis=0)
#     ax.plot(*mean_traj.T, color=PALETTE[name], lw=2.5, alpha=1.0)
#
#     # Stimulus onset marker
#     onset_idx = np.argmin(np.abs(times))  # time=0
#     ax.scatter(*mean_traj[onset_idx], color='k', s=30, zorder=5)
#
#     ax.set_title(['0-back', '1-back', '2-back'][cond], fontsize=10)
#     ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
#
# plt.savefig('../../figures/fig1_latent_trajectories.pdf', bbox_inches='tight')

print('Figure 1 template ready.')

In [ ]:
# ─── Statistical testing ──────────────────────────────────────────────────────
# Key tests:
#
# 1. θ_min: Is it significantly lower for 2-back than 0-back during maintenance?
#    Use paired t-test across subjects (n=4 — may need permutation test)
#
# 2. Q(t): Is it significantly higher for targets than non-targets in 2-back?
#    Use Wilcoxon signed-rank test across subjects
#
# 3. LQR energy: Does rescue energy correlate with pre-trial tangling Q?
#    Spearman correlation, bootstrapped CI
#
# NOTE on n=4: This is a small-n study. You MUST:
#   a) Use non-parametric tests where possible
#   b) Report effect sizes (Cohen's d), not just p-values
#   c) Use permutation testing for the main results
#   d) Acknowledge the limitation explicitly in the Discussion

def permutation_test(x, y, n_perms=1000, statistic=np.mean):
    """
    Permutation test for the difference between two groups.
    WHY: For small n, parametric assumptions (normality) don't hold.
    Permutation tests make no distributional assumptions.
    """
    observed = statistic(x) - statistic(y)
    pooled   = np.concatenate([x, y])
    n_x      = len(x)
    null_dist = []
    rng = np.random.default_rng(42)
    for _ in range(n_perms):
        shuffled = rng.permutation(pooled)
        null_dist.append(statistic(shuffled[:n_x]) - statistic(shuffled[n_x:]))
    p_val = np.mean(np.abs(null_dist) >= np.abs(observed))
    return observed, np.array(null_dist), p_val


print('Statistical testing functions ready.')

---
## 4. The Electrode Capacity Experiment (Pillar 6)

**The question:** How many electrodes do you actually need to capture WM dynamics?

This is a channel capacity question. With 40 electrodes, you have a communication channel between the neural tissue and your model. But most channels are redundant.

**Experiment:** Subsample electrodes: 2, 4, 8, 16, 32, all. For each subset:
1. Train VAE on the subset
2. Measure: how well do the latent trajectories decode condition (0/1/2-back)?
3. Plot decoding accuracy vs. electrode count

**Expected result:** Saturation around 8-16 electrodes — most information about WM dynamics is captured well before you use all channels. This is the effective capacity of the electrode array for this cognitive process.

**Why this matters:** In a real implanted BCI, electrode count is costly (surgery, tissue damage, signal stability). Knowing the saturation point informs device design. And the rate of saturation tells you about the effective dimensionality of the cognitive process — the same information you get from the scree plot, but from the engineering rather than modeling side.

---
## ✏️ Final Exercises

### A — Make Figure 2 Publication Quality
Rules:
- No chartjunk (no unnecessary borders, gridlines, tick marks)
- Error bands visible but not overwhelming
- Axes labeled with units
- Legend outside the plot area
- Colorblind-safe palette: use ColorBrewer or matplotlib's 'tab10'
- Size: 7" × 3.5" (fits Nature double-column)

### B — Write the Results Section
Structure:
1. "We trained a sequential VAE on high-gamma power and found [VAE result]."
2. "Figure 1 shows [trajectory result]."
3. "To test the geometric collapse hypothesis, we computed [metric]. Figure 2A shows [result]."
4. "Trajectory tangling (Figure 2B) showed [result]."
5. "Finally, we simulated a minimum-energy LQR rescue (Figure 3)..."

**Target:** 400 words, no jargon that isn't defined, one sentence per result.

### C — Identify the Two Biggest Limitations
Write a paragraph on each:
1. The most important methodological limitation (not the n=4 — that's obvious)
2. The most important conceptual limitation (what does the model assume that the biology might violate?)

*Identifying your own limitations before a reviewer does is a professional skill.*

### D — The Extension That Would Make This a Full Paper
In one paragraph: what would you need to do to turn this into a 6-8 figure journal paper? Be specific about experiments, datasets, and analyses.

---
## You Are Done With the Anchor Project

What you have built:
- A complete iEEG preprocessing and analysis pipeline
- A sequential VAE for latent trajectory extraction
- Principal angle and tangling analysis of WM dynamics  
- DMD-based dynamical system identification
- LQR rescue simulation with energy-accuracy tradeoff
- Three publication-quality figures
- Cross-subject validation on 4 subjects

What pillars you have strengthened:
1. Dynamical Systems Modeling ████████ Intermediate
2. System Identification ████░░░░ Beginner-Intermediate
3. Closed-Loop Control ██████░░ Intermediate
4. Neural Interface Engineering ████████ Intermediate
6. Information Theory ████░░░░ Beginner-Intermediate

What you now do:
- Extend to Sternberg WM dataset (Stanford mirror: exhibits.stanford.edu) — richer task design
- Start Tedrake 6.832 (Underactuated Robotics) — MPC and advanced control
- Email Tier 1 labs with this repo and a 1-page summary of Figure 2